# **Streaming Agents**

## **What's Covered?**
1. Streaming Agent's Response
    - Streaming Agent Progress
    - Streaming LLM Response
    - Streaming Multiple Modes

Interactive applications like chatbots or customer service agents can suffer from high latency. To get data to the user as soon as it is available you can deploy streaming. There are several streaming modes frequently used with the LangChain agents. 
1. Messages: Stream token by token as soon as they are produced by the LLM
2. Values: Return state after each step, for eg after the reasoning step or the tool call step, etc...
3. Custom: Helps stream data from the tools
4. Multiple sources

## **Streaming Agent's Response**

LangChain’s streaming system lets you surface live feedback from agent runs to your application.

Following is possible with LangChain Streaming:
1. Stream agent progress
2. Stream LLM tokens
3. Stream multiple modes
4. Stream custom updates (Check LangChain docs)

**Syntax**
```python
for chunk in agent.stream(  
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="updates",
):
    # do something
    pass
```

### **Streaming Agent Progress**

To stream agent progress, use the stream or astream methods with **stream_mode="updates"**. This emits an event after every agent step.

**Note:** Emit only the node or task names and updates returned by the nodes or tasks after each step. If multiple updates are made in the same step (e.g. multiple nodes are run) then those updates are emitted separately.

In [1]:
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

openai_chat_model = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model="gpt-4o-mini",
    temperature=0.0
)

In [2]:
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"

agent = create_agent(
    model=openai_chat_model,
    tools=[get_weather],
)

### **No Streaming**

In [3]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather in SF?"}]
})

In [6]:
response["messages"][-1].content

'The weather in San Francisco is always sunny!'

### **stream_mode="values"**

Value streaming mode returns data after each step in the agent loop. So we see updates after a model call, or a tool call, etc...

In [10]:
for chunk in agent.stream(  
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="values",
):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

What is the weather in SF?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_5arqwvI8iB4XPCKmRyrD6pqT)
 Call ID: call_5arqwvI8iB4XPCKmRyrD6pqT
  Args:
    city: San Francisco
================================= Tool Message =================================
Name: get_weather

It's always sunny in San Francisco!
================================== Ai Message ==================================

The weather in San Francisco is always sunny!


### **stream_mode="messages"**

Messages stream data token by token. This mode produce the lowest latency possible for the end user. This is perfect for interactive applications like chatbots or customer support agents. 

To stream tokens as they are produced by the LLM, use stream_mode="messages". Below you can see the output of the agent streaming tool calls and the final response.

Notice that, in the following output, it prints the tool message first and later the AI Message token by token.

In [18]:
for token, metadata in agent.stream(  
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="messages",
):
    if token.content:
        print(token.content, end="\n")

It's always sunny in San Francisco!
The
 weather
 in
 San
 Francisco
 is
 always
 sunny
!


### **stream_mode="custom"**

Tools can stream too. To stream updates from tools as they are executed, you can use **get_stream_writer**.

In [21]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()  
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"

agent = create_agent(
    model=openai_chat_model,
    tools=[get_weather],
)

In [22]:
for chunk in agent.stream(  
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="custom",
):
    print(chunk, end="\n")

Looking up data for city: San Francisco
Acquired data for city: San Francisco


### **stream_mode="updates"**

To stream agent progress, use the stream or astream methods with stream_mode="updates". This emits an event after every agent step.

In [20]:
for chunk in agent.stream(  
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'call_ZkX5OQWBd1NEMLU4vvAJSmGb'}]
step: tools
content: [{'type': 'text', 'text': "It's always sunny in San Francisco!"}]
step: model
content: [{'type': 'text', 'text': 'The weather in San Francisco is always sunny!'}]


### **Streaming LLM Tokens**

To stream tokens as they are produced by the LLM, use **stream_mode="messages"**. Below you can see the output of the agent streaming tool calls and the final response.

**Note:** Emit LLM messages token-by-token together with metadata for any LLM invocations inside nodes or tasks. Will be emitted as 2-tuples (LLM token, metadata).

In [ ]:
for token, metadata in agent.stream(  
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="messages",
):
    print(f"node: {metadata['langgraph_node']}")
    print(f"content: {token.content_blocks}")
    print("\n")

In [ ]:
for token, metadata in agent.stream(  
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="messages",
):
    if token.content:
        print(token.content, end="", flush=True)

### **Stream Multiple Modes**

You can specify multiple streaming modes by passing stream mode as a list: `stream_mode=["updates", "custom"]`. The streamed outputs will be tuples of `(mode, chunk)` where mode is the name of the stream mode and chunk is the data streamed by that mode.

In [ ]:
from langchain.messages import AIMessageChunk, AnyMessage, AIMessage, ToolMessage

def _render_message_chunk(token: AIMessageChunk) -> None:
    if token.text:
        print(token.text, end="|")
    if token.tool_call_chunks:
        print(token.tool_call_chunks)

def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Tool calls: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Tool response: {message.content_blocks}")        

for stream_mode, data in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in Boston?"}]},
    stream_mode=["messages", "updates"],  
):
    if stream_mode == "messages":
        token, metadata = data
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)
    if stream_mode == "updates":
        for source, update in data.items():
            if source in ("model", "tools"):
                _render_completed_message(update["messages"][-1])